In [1]:
import os
os.chdir('application')

In [ ]:
# !pip install hf_xet
# !pip install -U langchain-huggingface
# !pip install -U accelerate

In [2]:
from ingest.pdf_with_tables import extract_pdf_content

In [3]:
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\PAVILION\miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import re

In [6]:
# 1. Initialize the Hugging Face Embedding Model for Indonesian
def get_huggingface_embeddings():
    """Initializes and returns a Hugging Face embedding model for Indonesian."""
    model_name = "LazarusNLP/all-indo-e5-small-v4"
    model_kwargs = {'device': 'cpu'}  # Use 'cuda' for GPU
    encode_kwargs = {'normalize_embeddings': True}
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

In [7]:
embeddings = get_huggingface_embeddings()

In [10]:
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="percentile", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.7 # Adjust this value to control chunk size
)

In [5]:
pdf_pages = extract_pdf_content('../assets/CP-Cuti-dan-Izin-Khusus.pdf')

In [31]:
pdf_pages[0]

{'page': 1,
 'narrative_text': 'CUTI DAN IZIN KHUSUS\nDEFINISI\n● CUTI adalah hak untuk tidak melakukan pekerjaan dalam periode tertentu yang\ndiberikan kepada Karyawan sesuai dengan ketentuan yang ditetapkan oleh Perusahaan\n2\ndengan mempertimbangkan ketentuan perundangan yang berlaku.\n● KUARTAL adalah periode waktu selama 3 (tiga) bulan berturut-turut. 3\n● PERUSAHAAN adalah seluruh entitas-entitas legal yang berdasarkan keputusan\npimpinan tertinggi atau direksi KPN Group termasuk dalam ruang lingkup KPN\nCorporation. 4\n●\nTANGGAL MASUK KERJA (TMK) adalah tanggal efektif karyawan mulai bekerja di\nPerusahaan. 5\n● IZIN KHUSUS adalah kesempatan yang diberikan Perusahaan untuk tidak melakukan\npekerjaan dengan tetap menerima upah sejumlah waktu tertentu karena adanya\nkejadian-kejadian spesifik yang ditentukan dalam peraturan perundangan yang berlaku\n6\ndi Indonesia.\n● CUTI TAHUNAN adalah Cuti dengan menerima upah yang secara otomatis diberikan\nkepada Karyawan setelah Karyawan m

In [ ]:
print(pdf_pages[1]['narrative_text'])

● REMOTE AREA adalah area atau lokasi kerja yang ditetapkan Manajemen sebagai
lokasi yang jauh dari pemukiman warga atau tidak memiliki transportasi umum yang
memadai. 17
18
● SITE adalah kantor cabang dari Business Unit.
● PERJALANAN DINAS adalah perjalanan yang dilakukan oleh karyawan ke luar lokasi
kerja dalam rangka pelaksanaan tugas atau kepentingan Perusahaan. 19
●
SURAT PERINTAH PERJALANAN DINAS (SPPD) adalah surat digital atau non-digital
yang dibuat dan disetujui Atasan sebagai bentuk pemberian tugas bagi karyawan untuk
20
melakukan Perjalanan Dinas.
CUTI TAHUNAN
● Diberikan Kepada:
○ 21
Karyawan Kontrak: Diberikan 1 hari kerja setiap melewati 1 bulan masa kerja.
○ Karyawan Tetap: Diberikan per Kuartal. Setelah bekerja 3 bulan berturut-turut,
22
hak cuti akan muncul sebanyak 3 hari kerja.
● 23
Jumlah Kuota: 15 hari.
● Jumlah Maksimal Dalam 1 Pengajuan: 12 hari. 24
● Pengajuan dan Penggunaan Cuti Tahunan:
○ 25
Dapat dilakukan per ½ (setengah) hari kerja atau 1 (satu) hari kerja

In [11]:
# 4. Create the chunks
docs = text_splitter.split_text(pdf_pages[1]['narrative_text'])


# 5. View the results
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc)
    print("\n" + "="*20 + "\n")

--- Chunk 1 ---
● REMOTE AREA adalah area atau lokasi kerja yang ditetapkan Manajemen sebagai
lokasi yang jauh dari pemukiman warga atau tidak memiliki transportasi umum yang
memadai.


--- Chunk 2 ---
17
18
● SITE adalah kantor cabang dari Business Unit.


--- Chunk 3 ---
● PERJALANAN DINAS adalah perjalanan yang dilakukan oleh karyawan ke luar lokasi
kerja dalam rangka pelaksanaan tugas atau kepentingan Perusahaan.


--- Chunk 4 ---
19
●
SURAT PERINTAH PERJALANAN DINAS (SPPD) adalah surat digital atau non-digital
yang dibuat dan disetujui Atasan sebagai bentuk pemberian tugas bagi karyawan untuk
20
melakukan Perjalanan Dinas. CUTI TAHUNAN
● Diberikan Kepada:
○ 21
Karyawan Kontrak: Diberikan 1 hari kerja setiap melewati 1 bulan masa kerja.


--- Chunk 5 ---
○ Karyawan Tetap: Diberikan per Kuartal.


--- Chunk 6 ---
Setelah bekerja 3 bulan berturut-turut,
22
hak cuti akan muncul sebanyak 3 hari kerja.


--- Chunk 7 ---
● 23
Jumlah Kuota: 15 hari.


--- Chunk 8 ---
● Jumlah Maksimal Dal

Bikin titik dua dan header ke "previous header"

In [29]:
def extract_uppercase_words(text):
    # Find all words with 4 or more uppercase letters
    # Find sequences of uppercase letters and spaces
    phrases = re.findall(r'[A-Z\s]{4,}', text)

    # Clean up the list to get meaningful phrases
    capital_phrases = [phrase.strip() for phrase in phrases if phrase.strip() and not phrase.isspace()]
    return capital_phrases

In [30]:
extract_uppercase_words(pdf_pages[1]['narrative_text'])

['REMOTE AREA',
 'SITE',
 'PERJALANAN DINAS',
 'SURAT PERINTAH PERJALANAN DINAS',
 'SPPD',
 'CUTI TAHUNAN']

In [16]:
# The text starts by talking about Jakarta's history and then shifts to modern transportation.
jakarta_text = (
    "Jakarta, saaecara resmi bernama Daerah Khusus Ibukota Jakarta, adalah ibu kota negara dan kota terbesar di Indonesia. "
    "Terletak di pesisir bagian barat laut Pulau Jawa, kota ini telah menjadi pusat ekonomi, budaya, dan politik selama berabad-abad. "
    "Sejarahnya dimulai pada abad ke-4 sebagai sebuah pelabuhan perdagangan Hindu bernama Sunda Kelapa. "
    "Pada tahun 1527, Fatahillah dari Kesultanan Demak menaklukkan pelabuhan tersebut dan menamainya Jayakarta. "
    # --- The semantic chunker should identify a topic shift here ---
    "Sistem transportasi modern di Jakarta sangat bergantung pada bus dan kereta api. "
    "TransJakarta adalah sistem bus rapid transit (BRT) yang menjadi tulang punggung transportasi umum. "
    "Selain itu, MRT Jakarta dan LRT Jakarta juga telah beroperasi untuk mengurangi kemacetan lalu lintas yang parah di kota ini."
)


# 4. Create the chunks
docs = text_splitter.create_documents([jakarta_text])


# 5. View the results
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print("\n" + "="*20 + "\n")

--- Chunk 1 ---
Jakarta, secara resmi bernama Daerah Khusus Ibukota Jakarta, adalah ibu kota negara dan kota terbesar di Indonesia. Terletak di pesisir bagian barat laut Pulau Jawa, kota ini telah menjadi pusat ekonomi, budaya, dan politik selama berabad-abad.


--- Chunk 2 ---
Sejarahnya dimulai pada abad ke-4 sebagai sebuah pelabuhan perdagangan Hindu bernama Sunda Kelapa. Pada tahun 1527, Fatahillah dari Kesultanan Demak menaklukkan pelabuhan tersebut dan menamainya Jayakarta. Sistem transportasi modern di Jakarta sangat bergantung pada bus dan kereta api.


--- Chunk 3 ---
TransJakarta adalah sistem bus rapid transit (BRT) yang menjadi tulang punggung transportasi umum. Selain itu, MRT Jakarta dan LRT Jakarta juga telah beroperasi untuk mengurangi kemacetan lalu lintas yang parah di kota ini.




In [ ]:
import os
from langchain_experimental.text_splitter import SemanticChunker






# 2. Instantiate the Semantic Chunker
# The `breakpoint_threshold_type` tells the chunker how to find the topic changes.
# "percentile" is a great default. It means we'll split at the 95th percentile
# of similarity differences. In simple terms, it splits where the topic change
# is most significant.
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="percentile", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.95 # Adjust this value to control chunk size
)


# 3. Prepare an example text with a clear topic shift
